# GPU Execution Notebook — Mechanistic Interpretability

This notebook runs the full pipeline on Colab GPU:
1. Setup & install
2. Generate datasets
3. Capture activations (8 layers: 4,8,12,16,20,24,28,32)
4. Train SAEs
5. Build attribution graphs
6. Run intervention experiments (full-knockout diagnostic, scan, inhibit, swap)

**Runtime settings**: GPU (any tier), High-RAM recommended.

In [ ]:
# Step 1: Mount Google Drive and extract project
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os

# Set to True if you uploaded the project.zip directly to Colab files sidebar:
uploaded_directly_to_colab = False

if uploaded_directly_to_colab:
    zip_path = "/content/project.zip"
else:
    zip_path = "/content/drive/MyDrive/mphil-project/project.zip"

if os.path.exists(zip_path):
    print(f"Extracting {zip_path}...")
    !unzip -q -o {zip_path} -d /content/
    os.chdir('/content')
    print(f"Current working directory: {os.getcwd()}")
    !ls -l
else:
    print(f"ERROR: Could not find {zip_path}.")
    print("Please upload 'project.zip' to Google Drive or the Colab files sidebar.")

In [ ]:
# Step 2: Install Dependencies
!pip install --upgrade "transformers>=4.51.0" accelerate
!pip install -e .

In [ ]:
# Step 3: Generate Datasets
!python data/generate_datasets.py --capitals
print("\nDataset files:")
!ls -lh data/*.csv

In [ ]:
# Step 4: Capture Activations for 8 layers
# This takes ~5-10 minutes on a T4/V100/A100
!python src/capture_activations.py \
  --output-dir mechanistic_data \
  --layers 4 8 12 16 20 24 28 32 \
  --seed 787

In [ ]:
# Step 5: Train SAEs on all 8 layers
# This takes ~10-20 minutes on GPU depending on hardware
!python src/train.py --config configs/sae_config.yaml
print("\nSAE checkpoint files:")
!ls -lh mechanistic_data/sae_checkpoints/

---
## Phase 5: Attribution Graphs

Build attribution graphs for different prompts. The graph identifies which SAE features
are most important for predicting the next token.

In [ ]:
# Graph 1: Capital of state (Texas/Dallas -> Austin)
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target "Austin" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/dallas_graph.json \
  --output-html outputs/dallas_graph.html \
  --output-mermaid outputs/dallas_graph.md

In [ ]:
# Graph 2: Capital of country (Afghanistan/Kandahar -> Kabul)
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target "Kabul" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/kabul_graph.json \
  --output-html outputs/kabul_graph.html \
  --output-mermaid outputs/kabul_graph.md

In [ ]:
# Graph 3: A lower-confidence example — better for showing intervention effects
# Delaware/Wilmington -> Dover (model may be less confident here)
!python src/attribution_graph.py \
  --prompt "Fact: The capital of the state containing Wilmington is named" \
  --target "Dover" \
  --top-k-nodes 20 --top-k-edges 30 \
  --output-json outputs/dover_graph.json \
  --output-html outputs/dover_graph.html \
  --output-mermaid outputs/dover_graph.md

---
## Phase 6: Intervention Experiments

### Diagnostic 1: Full MLP Knockout
First, verify the hooked layers actually matter by zeroing out their MLP outputs entirely.
If this doesn't change the prediction, the layers we chose are uninvolved.

In [ ]:
# Full knockout: proves the layers matter (upper bound on intervention effect)
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin" \
  --full-knockout \
  --output outputs/knockout_dallas.json

In [ ]:
# Full knockout on Kabul prompt
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target-token "Kabul" \
  --full-knockout \
  --output outputs/knockout_kabul.json

### Diagnostic 2: Progressive Ablation Scan

Ablate features progressively (top-10, 25, 50, 100, ALL from the graph)
to see how many features need to be removed before the logit shifts.

In [ ]:
# Scan: progressive ablation using features from the attribution graph
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin" \
  --graph-json outputs/dallas_graph.json \
  --scan \
  --output outputs/scan_dallas.json

In [ ]:
# Scan on Kabul prompt
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target-token "Kabul" \
  --graph-json outputs/kabul_graph.json \
  --scan \
  --output outputs/scan_kabul.json

In [ ]:
# Scan on Dover prompt (likely lower confidence -> easier to observe effects)
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Wilmington is named" \
  --target-token "Dover" \
  --graph-json outputs/dover_graph.json \
  --scan \
  --output outputs/scan_dover.json

### Experiment 1: Targeted Inhibition

Use the features identified by the attribution graph. The `--graph-json` flag
automatically extracts all active features from the pruned graph.

In [ ]:
# Inhibition with ALL features from the attribution graph
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Austin" \
  --graph-json outputs/dallas_graph.json \
  --output outputs/inhibit_dallas_full.json

In [ ]:
# Inhibition on Kabul
!python src/intervention.py \
  --mode inhibit \
  --prompt "Fact: The capital of the country containing Kandahar is named" \
  --target-token "Kabul" \
  --graph-json outputs/kabul_graph.json \
  --output outputs/inhibit_kabul_full.json

### Experiment 2: Activation Swap-In

Swap features from a source prompt (where model predicts X) into a target prompt
(where model predicts Y). If the swapped features encode X's identity, the target
prompt should shift toward predicting X.

In [ ]:
# Swap: Take features from Oakland (Sacramento) and swap into Dallas (Austin)
# Hypothesis: if location-identity features are swapped, Dallas prompt shifts toward Sacramento
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Oakland is named" \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --graph-json outputs/dallas_graph.json \
  --target-token "Sacramento, Austin" \
  --output outputs/swap_oakland_to_dallas.json

In [ ]:
# Swap: Full latent swap (no --features, swaps everything) — strongest signal
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the state containing Oakland is named" \
  --prompt "Fact: The capital of the state containing Dallas is named" \
  --target-token "Sacramento, Austin" \
  --output outputs/swap_full_oakland_to_dallas.json

In [ ]:
# Swap across countries: Kandahar (Kabul) -> Berlin prompt
!python src/intervention.py \
  --mode swap \
  --source-prompt "Fact: The capital of the country containing Kandahar is named" \
  --prompt "Fact: The capital of the country containing Munich is named" \
  --graph-json outputs/kabul_graph.json \
  --target-token "Kabul, Berlin" \
  --output outputs/swap_kandahar_to_munich.json

---
## Save Outputs to Drive

In [ ]:
# Copy all outputs to Google Drive for persistence
import shutil
drive_out = '/content/drive/MyDrive/mphil-project/outputs'
os.makedirs(drive_out, exist_ok=True)
for f in os.listdir('/content/outputs'):
    src = f'/content/outputs/{f}'
    if os.path.isfile(src):
        shutil.copy2(src, drive_out)
        print(f'Copied {f}')

# Also copy SAE checkpoints
drive_sae = '/content/drive/MyDrive/mphil-project/mechanistic_data/sae_checkpoints'
os.makedirs(drive_sae, exist_ok=True)
sae_dir = '/content/mechanistic_data/sae_checkpoints'
if os.path.isdir(sae_dir):
    for f in os.listdir(sae_dir):
        shutil.copy2(f'{sae_dir}/{f}', drive_sae)
    print(f'\nCopied SAE checkpoints to {drive_sae}')
print('Done!')

---
## Results Summary

After running, report back the following information:

1. **Full knockout results**: did zeroing out all 8 MLP layers change the top prediction?
   - Copy the printed output from the `--full-knockout` cells.

2. **Scan results**: at what level of progressive ablation does the logit shift become noticeable?
   - Copy the logit deltas printed by the `--scan` cells.

3. **Swap results**: did the full latent swap shift the prediction toward the source capital?
   - Copy the target token probabilities from the swap cells.

4. **Any errors or unexpected behavior** during execution.

This information tells us:
- If knockout has no effect → the layers are wrong or the model uses other pathways
- If scan shows gradual decline → features are working correctly but distributed
- If full swap shifts output → SAE features encode identity information correctly